In [3]:
import requests

# Configurações da API do Mentor
B365_TOKEN = "248558-x464EYT2kttm4b"
B365_BASE_URL = "https://api.b365api.com/v3/events/inplay"

async def monitorar_jogo_ao_vivo(event_id):
    # 1. Busca os dados reais da APIBet
    # Nota: Usamos o /event/view para pegar o detalhe de um jogo específico
    url = f"https://api.b365api.com/v1/event/view?token={B365_TOKEN}&event_id={event_id}"
    
    try:
        response = requests.get(url)
        data = response.json()
        
        if data.get("success") != 1:
            return "Erro ao buscar dados da API"

        # 2. Extração dos Dados para as 9 Features do seu Oráculo
        event = data['results'][0]
        stats = event.get('stats', {})
        
        # Mapeamento para o seu modelo (Exemplo de extração da B365)
        # B365 costuma mandar listas: [casa, fora]
        ataques_casa = int(stats.get('dangerous_attacks', [0, 0])[0])
        escanteios_casa = int(stats.get('corners', [0, 0])[0])
        
        # Como a API não dá X e Y da bola, simulamos 'Zona de Perigo' 
        # se o último evento for de ataque
        x_simulado = 110.0  # Assumimos que o Oráculo só analisa lances de ataque
        y_simulado = 40.0
        
        # 3. Monta o Array de 9 Features (A ordem que você treinou!)
        # [x, y, dist, angulo, pressao, acel, progresso, cantos, ataques]
        # (Cálculos de dist e angulo devem ser feitos aqui com x_simulado)
        
        return {
            "time_casa": event['home']['name'],
            "time_fora": event['away']['name'],
            "stats_realtime": {
                "ataques_perigosos": ataques_casa,
                "escanteios": escanteios_casa
            }
        }
    except Exception as e:
        return f"Falha na conexão: {e}"

In [ ]:
# Como a B365 costuma entregar: stats = {"dangerous_attacks": [57, 40], "corners": [8, 3]}
def mapear_para_ia(data_b365):
    # Pegamos os dados do time da CASA (índice 0)
    ataques = data_b365['stats']['dangerous_attacks'][0]
    escanteios = data_b365['stats']['corners'][0]
    
    # Simulamos a geometria de um ataque perigoso padrão
    x, y = 110.0, 40.0 
    dist = 10.0 # Calculado via Pitágoras
    ang = 45.0  # Calculado via Lei dos Cossenos
    
    # Montamos a lista EXATAMENTE na ordem do seu treino
    return [x, y, dist, ang, 0.7, 5.0, 2.0, escanteios, ataques]




In [5]:
import requests
import json
from rich import print as rprint # Se não tiver, 'pip install rich'

# --- CONFIGURAÇÃO ---
TOKEN = "248558-x464EYT2kttm4b" # Cole o token que o mentor passou
URL = "https://api.b365api.com/v3/events/inplay"
PARAMS = {
    "token": TOKEN,
    "sport_id": "1"  # 1 é Futebol
}

def capturar_json():
    print(f"🚀 Chamando a B365API...")
    
    try:
        # Faz a requisição enviando o token nos parâmetros da URL (Query Params)
        response = requests.get(URL, params=PARAMS, timeout=10)
        
        # Transforma em dicionário Python
        dados = response.json()
        
        print(f"📡 Status Code: {response.status_code}")
        
        if dados.get("success") == 1:
            print("✅ SUCESSO! Veja o JSON abaixo:")
            # O 'rprint' da biblioteca 'rich' deixa o JSON colorido e identado
            rprint(dados) 
            
            # Opcional: Salva num arquivo para você abrir no VS Code
            with open("resposta_real.json", "w", encoding="utf-8") as f:
                json.dump(dados, f, indent=4, ensure_ascii=False)
            print("\n📂 Salvei também em 'resposta_real.json' pra você analisar com calma.")
            
        else:
            print("❌ A API respondeu, mas deu ERRO:")
            rprint(dados)

    except Exception as e:
        print(f"💥 ERRO CRÍTICO na conexão: {e}")

if __name__ == "__main__":
    capturar_json()

🚀 Chamando a B365API...
📡 Status Code: 200
✅ SUCESSO! Veja o JSON abaixo:


{
    'success': 1,
    'pager': {'page': 1, 'per_page': 1000, 'total': 60},
    'results': [
        {
            'id': '11628095',
            'sport_id': '1',
            'time': '1774625400',
            'time_status': '3',
            'league': {'id': '28457', 'name': 'Qatar Reserve League', 'cc': 'qa'},
            'home': {'id': '186131', 'name': 'Al-Arabi Doha U23', 'image_id': '1280741', 'cc': 'qa'},
            'away': {'id': '194450', 'name': 'Al-Sailiya SC U23', 'image_id': '524622', 'cc': 'qa'},
            'ss': '3-3',
            'scores': {'2': {'home': '3', 'away': '3'}, '1': {'home': '1', 'away': '1'}},
            'bet365_id': '191870312',
            'timer': {'tm': 90, 'ts': 0, 'tt': '0', 'ta': 0, 'md': 1},
            'stats': {
                'attacks': ['83', '91'],
                'corners': ['8', '2'],
                'corner_f': ['8', '2'],
                'corner_h': ['6', '0'],
                'dangerous_attacks': ['72', '46'],
                'goals': ['3', '3'],
                'off_target': ['7', '4'],
                'on_target': ['9', '8'],
                'penalties': ['0', '0'],
                'possession_rt': ['50', '50'],
                'redcards': ['1', '0'],
                'substitutions': ['5', '3'],
                'yellowcards': ['2', '1']
            }
        },
        {
            'id': '11628096',
            'sport_id': '1',
            'time': '1774625400',
            'time_status': '3',
            'league': {'id': '28457', 'name': 'Qatar Reserve League', 'cc': 'qa'},
            'home': {'id': '234130', 'name': 'Al Shahaniya U23', 'image_id': '1280753', 'cc': 'qa'},
            'o_home': {'id': '303629', 'name': 'Al-Shahaniya SC U23', 'image_id': '1280753', 'cc': None},
            'away': {'id': '186275', 'name': 'Al-Ahli Doha U23', 'image_id': '1280739', 'cc': 'qa'},
            'ss': '1-0',
            'scores': {'2': {'home': '1', 'away': '0'}, '1': {'home': '0', 'away': '0'}},
            'bet365_id': '191870298',
            'timer': {'tm': 90, 'ts': 0, 'tt': '0', 'ta': 0, 'md': 1},
            'stats': {
                'attacks': ['83', '76'],
                'corners': ['8', '5'],
                'corner_f': ['8', '5'],
                'corner_h': ['2', '4'],
                'dangerous_attacks': ['63', '64'],
                'goals': ['1', '0'],
                'off_target': ['4', '6'],
                'on_target': ['4', '5'],
                'penalties': ['0', '0'],
                'possession_rt': ['54', '46'],
                'redcards': ['0', '0'],
                'substitutions': ['3', '4'],
                'yellowcards': ['2', '0']
            }
        },
        {
            'id': '11628097',
            'sport_id': '1',
            'time': '1774625400',
            'time_status': '1',
            'league': {'id': '28457', 'name': 'Qatar Reserve League', 'cc': 'qa'},
            'home': {'id': '186129', 'name': 'Umm Salal U23', 'image_id': '71084', 'cc': 'qa'},
            'away': {'id': '302033', 'name': 'Al-Wakrah SC U23', 'image_id': '1280757', 'cc': 'qa'},
            'ss': '0-1',
            'scores': {'1': {'home': '0', 'away': '1'}, '2': {'home': '0', 'away': '1'}},
            'bet365_id': '191870314',
            'timer': {'tm': 95, 'ts': 47, 'tt': '1', 'ta': 9, 'md': 1},
            'stats': {
                'attacks': ['100', '65'],
                'corners': ['4', '1'],
                'corner_h': ['2', '1'],
                'dangerous_attacks': ['96', '46'],
                'goals': ['0', '1'],
                'off_target': ['5', '4'],
                'on_target': ['2', '2'],
                'penalties': ['0', '0'],
                'possession_rt': ['64', '36'],
                'redcards': ['0', '0'],
                'substitutions': ['5', '5'],
                'yellowcards': ['5', '4']
            }
        },
        {
            'id': '11596321',
            'sport_id': '1',
            'time': '1774625700',
            'time_status'


📂 Salvei também em 'resposta_real.json' pra você analisar com calma.


In [7]:
import json
import numpy as np
from rich.console import Console
from rich.table import Table

console = Console()

# --- DADOS QUE VOCÊ RECEBEU (JSON DA BETSAPI) ---
data_da_api = {
    "results": [
        {
            "home": {"name": "Al-Arabi Doha U23"},
            "away": {"name": "Al-Sailiya SC U23"},
            "ss": "3-3",
            "timer": {"tm": 90},
            "stats": {"dangerous_attacks": ["72", "46"], "corners": ["8", "2"]}
        },
        {
            "home": {"name": "Umm Salal U23"},
            "away": {"name": "Al-Wakrah SC U23"},
            "ss": "0-1",
            "timer": {"tm": 95},
            "stats": {"dangerous_attacks": ["96", "46"], "corners": ["4", "1"]}
        }
    ]
}

def simular_predicao_ia(ataques, cantos):
    """
    Simula o que o seu oraculo_model.predict_proba faria.
    Substitua esta lógica pela chamada real do seu modelo carregado!
    """
    # Exemplo de lógica simplificada para o teste:
    # (Ataques / 100) + (Cantos / 20) = Probabilidade
    prob = (ataques / 120) + (cantos / 25)
    return min(prob, 0.99) # Nunca passa de 99%

def exibir_dashboard_oraculo(json_bruto):
    table = Table(title="🔮 ORÁCULO LIVE - DASHBOARD DE PREDIÇÃO", show_header=True, header_style="bold magenta")
    
    table.add_column("Partida", style="dim", width=30)
    table.add_column("Placar", justify="center")
    table.add_column("Tempo", justify="center")
    table.add_column("Atq. Perigosos (C/V)", justify="center")
    table.add_column("Escanteios (C/V)", justify="center")
    table.add_column("PROB. GOL (ORÁCULO)", justify="right", style="bold green")

    for jogo in json_bruto['results']:
        home = jogo['home']['name']
        away = jogo['away']['name']
        placar = jogo['ss']
        minuto = jogo['timer']['tm']
        
        # Extração e conversão
        atq_c = int(jogo['stats']['dangerous_attacks'][0])
        atq_v = int(jogo['stats']['dangerous_attacks'][1])
        cnt_c = int(jogo['stats']['corners'][0])
        cnt_v = int(jogo['stats']['corners'][1])

        # AQUI VOCÊ CHAMA SUA IA PARA O TIME DA CASA
        # Na vida real: prob = oraculo_model.predict_proba([features])[0][1]
        prob_casa = simular_predicao_ia(atq_c, cnt_c)

        # Cor do alerta baseada na probabilidade
        cor_prob = "green"
        if prob_casa > 0.6: cor_prob = "yellow"
        if prob_casa > 0.8: cor_prob = "red"

        table.add_row(
            f"{home} vs {away}",
            placar,
            f"{minuto}'",
            f"{atq_c} / {atq_v}",
            f"{cnt_c} / {cnt_v}",
            f"[{cor_prob}]{prob_casa:.1%}[/{cor_prob}]"
        )

    console.print(table)

if __name__ == "__main__":
    exibir_dashboard_oraculo(data_da_api)

                                      🔮 ORÁCULO LIVE - DASHBOARD DE PREDIÇÃO                                      
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┓
┃ Partida                        ┃ Placar ┃ Tempo ┃ Atq. Perigosos (C/V) ┃ Escanteios (C/V) ┃ PROB. GOL (ORÁCULO) ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━┩
│ Al-Arabi Doha U23 vs           │  3-3   │  90'  │       72 / 46        │      8 / 2       │               92.0% │
│ Al-Sailiya SC U23              │        │       │                      │                  │                     │
│ Umm Salal U23 vs Al-Wakrah SC  │  0-1   │  95'  │       96 / 46        │      4 / 1       │               96.0% │
│ U23                            │        │       │                      │                  │                     │
└────────────────────────────────┴────────┴───────┴──────────────────────┴──────────────────┴─────────────────────┘